In [1]:
install.packages("readxl") # run this once to install the package

Installing package into ‘/srv/r’
(as ‘lib’ is unspecified)



In [2]:
library(dplyr)
library(readxl)
zev <- read_excel("data/New_ZEV_Sales_Last_updated_07-17-2026_ada.xlsx",
  sheet = "County")


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




# Part 1: The CEC Dashboard

### What are three different questions that can be answered by interacting this dashboard? Select three questions that you are truly curious about.

1. What is the county that had the smallest number of new EV purchases in a year? - 2026 Alpine County
2. What is the county with the most EV purchases in the year? - 2026 Los Angeles County
3. What is the most common model of EV? - 2026 Tesla Model Y

# Part 2: The Data Behind the Dashboard

#### After reading all three sheets of the Excel file as well as the documentation on the original dashboard site, state as precisely as possible: does this data record new ZEV sales? Or new purchases? Or new registrations? What types of transactions are included and what are excluded from this tally?

The documentation states that the Excel file documents new vehicle sales by examining DMV registration data. This is because there might be a new registration, which is a transfer of ownership not an acutal new vehicle. The data only includes new vehicle sales so no secondary market resales.

#### For the Country sheet in particular, what does every row refer to (i.e. what is the unit of observation)?

Every row in the County sheet refers to a the number of new registerd vehicles per county per quarter. As the data below shows the first line refers to a singular Tesla Roadster that was bought in the 3rd quarter of 2008 in Alameda County.

In [3]:
head(zev)

Data Year,Quarter,COUNTY,FUEL_TYPE,MAKE,MODEL,Number of Vehicles
<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<dbl>
2008,3,Alameda,Electric,Tesla,Roadster,1
2008,3,San Francisco,Electric,Tesla,Roadster,1
2008,3,San Mateo,Electric,Tesla,Roadster,3
2008,3,Santa Clara,Electric,Tesla,Roadster,4
2008,3,Sonoma,Electric,Tesla,Roadster,1
2008,4,Alameda,Electric,Tesla,Roadster,5


# Part 3: Checking the Dashboard

Find where on the Dashboard each of the following statistics are reported, then recreate them from the zev data using a dplyr pipeline. If there are any you cannot recreate, state why not.

In [4]:
cat("Cumulative light-duty ZEV sales through 2026:", sum(zev[["Number of Vehicles"]])) 

Cumulative light-duty ZEV sales through 2026: 2702646

In [5]:
zev_YTD_2026 <- zev |>
    filter(`Data Year` == 2026) |>
    pull(`Number of Vehicles`) |>
    sum()

cat("Year-to-date 2026 total sales:", zev_YTD_2026) 

Year-to-date 2026 total sales: 151525

In [6]:
YTD_Alameda_2026 <- zev |>
    filter(`Data Year` == 2026, `COUNTY` == "Alameda") |>
    pull(`Number of Vehicles`) |>
    sum()

cat("Year-to-date 2026 total sales for Alameda:", YTD_Alameda_2026) 

Year-to-date 2026 total sales for Alameda: 8754

In [7]:
YTD_BEV_2026 <- zev |>
    filter(`Data Year` == 2026, `FUEL_TYPE` == "Electric") |>
    pull(`Number of Vehicles`) |>
    sum()

YTD_PHEV_2026 <- zev |>
    filter(`Data Year` == 2026, `FUEL_TYPE` == "PHEV") |>
    pull(`Number of Vehicles`) |>
    sum()

YTD_FCEV_2026 <- zev |>
    filter(`Data Year` == 2026, `FUEL_TYPE` == "Hydrogen") |>
    pull(`Number of Vehicles`) |>
    sum()

cat("The 2026 total sales BEV:", YTD_BEV_2026, "\n")
cat("The 2026 total sales PHEV:", YTD_PHEV_2026, "\n") 
cat("The 2026 total sales FCEV:", YTD_FCEV_2026, "\n") 

The 2026 total sales BEV: 132453 
The 2026 total sales PHEV: 18901 
The 2026 total sales FCEV: 171 


In [8]:
cat("The year-to-date (YTD) Zev Share:", zev_YTD_2026/sum(zev[["Number of Vehicles"]])*100, "% \n")

The year-to-date (YTD) Zev Share: 5.606543 % 


In [9]:
# A table of counts of 2026 Tesla sales, split by model

tesla_YTD_2026 <- zev |>
    filter(`Data Year` == 2026, `MAKE` == "Tesla") |>
    group_by(`MODEL`) |>
    summarise(
        Total_Sales = sum(`Number of Vehicles`),
    )

tesla_YTD_2026

MODEL,Total_Sales
<chr>,<dbl>
Cybertruck,1499
Model 3,17827
Model S,1667
Model X,2809
Model Y,53682


In [10]:
sales_YTD_2026 <- zev |> 
    filter(`Data Year` == 2026) |> 
    group_by(MAKE, MODEL) |> 
    summarise(
        Models = sum(`Number of Vehicles`),
    )

sales_by_make <- sales_YTD_2026 |>
    group_split(MAKE)

sales_by_make

`summarise()` has regrouped the output.
ℹ Summaries were computed grouped by MAKE and MODEL.
ℹ Output is grouped by MAKE.
ℹ Use `summarise(.groups = "drop_last")` to silence this message.
ℹ Use `summarise(.by = c(MAKE, MODEL))` for per-operation grouping
  (`?dplyr::dplyr_by`) instead.
Warning message:
“Calling `group_split()` on a <grouped_df> ignores `...`. Please use
`group_by(..., .add = TRUE) |> group_split()`.”


MAKE,MODEL,Models
<chr>,<chr>,<dbl>
Alfa Romeo,Tonale Hybrid,13
Alfa Romeo,Tonale Tributo Italiano,1
MAKE,MODEL,Models
<chr>,<chr>,<dbl>
Aston Martin,Valhalla,16
MAKE,MODEL,Models
<chr>,<chr>,<dbl>
Audi,A6 Sportback e-tron,78
Audi,Q4 e-tron,45
